<a href="https://colab.research.google.com/github/JudithJacquet/TFG-Pronostico-Picos-SIN/blob/prueba_modelos/modelo_definitivo_walk_forward_semestral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo definitivo de demanda máxima diaria con XGBoost

Este cuaderno consolida la configuración elegida:

- Fechas agrupadas con la zona horaria local de Paraguay (America/Asuncion).
- Features térmicas, de calendario, humedad, rezagos de demanda 21/28 días y viento.
- Ventana móvil de entrenamiento de 10 años y reentrenamiento semestral.
- Optuna: entrenamiento hasta 2017 y validación 2018–2019.
- Corrección móvil del sesgo calibrada con 2018–2019 y evaluada en 2020–2022.

No se generan gráficos. Las métricas de 2020–2022 son retrospectivas: ese período ya se usó durante las pruebas de features. Las features meteorológicas del día objetivo se toman del dataset histórico; para una predicción operativa deben provenir de pronósticos disponibles ese día.

## Bloque 1 — Instalar librerías, importar y conectar Drive

In [1]:
!pip install -q holidays optuna xgboost openpyxl

import warnings
import pandas as pd
import numpy as np
import holidays
import optuna
import xgboost as xgb

from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error
from google.colab import drive, files

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

drive.mount('/content/drive')

print("Librerías listas y Google Drive conectado.")

Mounted at /content/drive
Librerías listas y Google Drive conectado.


## Bloque 2 — Cargar datos y crear variables diarias en hora paraguaya

Verifica CSV_PATH si el archivo está en otra carpeta de Drive. Los días se forman después de convertir los timestamps a America/Asuncion.

In [2]:
CSV_PATH = '/content/drive/MyDrive/Fabri/complete-dataset.csv'
ZONA_PARAGUAY = 'America/Asuncion'

df_hourly = pd.read_csv(CSV_PATH, index_col='DATETIME')

columnas_requeridas = [
    'SIN', 'T02M', 'ISHOLIDAY', 'RH2M', 'U10M', 'V10M'
]
faltantes = [
    columna for columna in columnas_requeridas
    if columna not in df_hourly.columns
]
if faltantes:
    raise ValueError(
        "Faltan columnas necesarias en el CSV: " + ", ".join(faltantes)
    )

# Interpreta los timestamps del archivo y los convierte a hora paraguaya.
df_hourly.index = (
    pd.to_datetime(df_hourly.index, utc=True)
      .tz_convert(ZONA_PARAGUAY)
)
df_hourly = df_hourly.sort_index()

# Agregación por día local de Paraguay.
agregados = df_hourly.resample('D').agg({
    'SIN': ['max'],
    'T02M': ['mean', 'max', 'min'],
    'ISHOLIDAY': ['max'],
    'RH2M': ['mean', 'max']
})
agregados.columns = [
    'SIN_max',
    'T02M_mean',
    'T02M_max',
    'T02M_min',
    'ISHOLIDAY_max',
    'RH2M_mean',
    'RH2M_max'
]

# Velocidad del viento a partir de sus componentes U10M y V10M.
viento_horario = np.sqrt(
    df_hourly['U10M'] ** 2 +
    df_hourly['V10M'] ** 2
)
viento_diario = viento_horario.resample('D').agg(['mean', 'max'])
viento_diario.columns = ['WIND10M_mean', 'WIND10M_max']

df_daily = agregados.join(viento_diario, how='left')

# Quitar la zona horaria después de agrupar; las fechas ya son locales.
df_daily.index = df_daily.index.tz_localize(None)
df_daily = df_daily.sort_index()

# Feriados de Paraguay y vísperas.
primer_anio = df_daily.index.min().year
ultimo_anio = df_daily.index.max().year
feriados_py = holidays.Paraguay(
    years=range(primer_anio - 1, ultimo_anio + 2)
)

df_daily['is_py_holiday'] = [
    int(fecha.date() in feriados_py)
    for fecha in df_daily.index
]
df_daily['is_holiday_eve'] = (
    df_daily['is_py_holiday']
    .shift(-1)
    .fillna(0)
    .astype(int)
)
df_daily['is_dec_24_31'] = (
    (df_daily.index.month == 12) &
    (df_daily.index.day.isin([24, 31]))
).astype(int)

# Indicador de receso escolar, con la misma heurística del experimento V4.
mes = df_daily.index.month
dia = df_daily.index.day
receso_verano = (
    mes.isin([12, 1, 2]) |
    ((mes == 11) & (dia >= 15))
)
receso_invierno = (
    (mes == 7) &
    (dia >= 10) &
    (dia <= 25)
)
df_daily['receso_escolar'] = (
    receso_verano | receso_invierno
).astype(int)

# Interacción temperatura-humedad.
df_daily['T02M_RH2M_interaction'] = (
    df_daily['T02M_mean'] *
    df_daily['RH2M_mean'] / 100.0
)

print(f"Primer día: {df_daily.index.min().date()}")
print(f"Último día: {df_daily.index.max().date()}")
print(f"Filas diarias: {len(df_daily)}")
display(df_daily.head())

Primer día: 2008-01-01
Último día: 2022-12-31
Filas diarias: 5479


,SIN_max,T02M_mean,T02M_max,T02M_min,ISHOLIDAY_max,RH2M_mean,RH2M_max,WIND10M_mean,WIND10M_max,is_py_holiday,is_holiday_eve,is_dec_24_31,receso_escolar,T02M_RH2M_interaction
DATETIME,,,,,,,,,,,,,,
2008-01-01,1251.0,26.728509,30.957381,22.969847,1.0,83.978795,96.798320,2.311934,3.205951,1,0,0,1,22.446280
2008-01-02,1440.6,27.539757,32.673443,22.823432,0.0,82.325824,97.481116,1.696893,2.285783,0,0,0,1,22.672332
2008-01-03,1458.0,28.175060,32.470740,24.350555,0.0,85.061181,97.481116,2.214298,3.443342,0,0,0,1,23.966038
2008-01-04,1405.0,27.460105,30.067234,24.802080,0.0,88.566013,98.481116,1.469866,2.265441,0,0,0,1,24.320320
2008-01-05,1379.0,27.778989,33.106973,23.371475,0.0,74.476612,94.635901,2.235454,3.940047,0,0,0,1,20.688850


## Bloque 3 — Definir la construcción final de features

Incluye la base V4, temperatura máxima/mínima/rango, humedad e interacción, rezagos 21/28 y viento. No incluye las features adicionales de día puente/post-feriado ni acumulación de calor que no quedaron en la configuración seleccionada.

In [3]:
def build_features_final(daily_data, trend_end):
    """Construye features con la tendencia ajustada solo hasta trend_end."""
    df_feat = daily_data.copy()

    # Índice temporal y tendencia lineal sin usar datos posteriores al corte.
    df_feat['time_idx'] = (
        df_feat.index - df_feat.index.min()
    ).days

    fit_mask = (
        (df_feat.index <= trend_end) &
        df_feat['SIN_max'].notna()
    )
    if fit_mask.sum() < 2:
        raise ValueError(
            f"No hay suficientes observaciones para ajustar la tendencia "
            f"hasta {trend_end}."
        )

    coef = np.polyfit(
        df_feat.loc[fit_mask, 'time_idx'].to_numpy(),
        df_feat.loc[fit_mask, 'SIN_max'].to_numpy(),
        1
    )

    df_feat['trend'] = (
        coef[0] * df_feat['time_idx'] + coef[1]
    )
    if (df_feat['trend'] <= 0).any():
        raise ValueError("La tendencia tiene valores cero o negativos.")

    df_feat['target_detrended'] = (
        df_feat['SIN_max'] / df_feat['trend']
    )

    # Calendario
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['month'] = df_feat.index.month
    df_feat['dayofyear'] = df_feat.index.dayofyear
    df_feat['is_weekend'] = (
        df_feat.index.dayofweek >= 5
    ).astype(int)
    df_feat['is_workday'] = (
        (df_feat['is_weekend'] == 0) &
        (df_feat['is_py_holiday'] == 0)
    ).astype(int)

    # Variables térmicas
    df_feat['T02M_range'] = (
        df_feat['T02M_max'] - df_feat['T02M_min']
    )
    df_feat['CDD_24'] = np.maximum(
        0, df_feat['T02M_mean'] - 24.0
    )
    df_feat['CDD_excess_sq'] = (
        np.maximum(0, df_feat['T02M_mean'] - 30.0) ** 2
    )
    df_feat['temp_workday'] = (
        df_feat['T02M_mean'] * df_feat['is_workday']
    )
    df_feat['T02M_diff1'] = df_feat['T02M_mean'].diff(1)
    df_feat['T02M_diff7'] = df_feat['T02M_mean'].diff(7)
    df_feat['temp_shock'] = (
        df_feat['T02M_diff1'] > 4.0
    ).astype(int)

    is_hot = (df_feat['T02M_mean'] > 31.0).astype(int)
    df_feat['extreme_heatwave'] = (
        is_hot.rolling(window=3).sum() == 3
    ).astype(int)

    # Rezagos del objetivo des-tendenciado
    for lag in [1, 2, 3, 7, 14, 21, 28]:
        df_feat[f'target_detrended_lag_{lag}'] = (
            df_feat['target_detrended'].shift(lag)
        )

    # Rezagos y promedio móvil de temperatura
    for lag in [1, 2, 3, 7]:
        df_feat[f'T02M_lag_{lag}'] = (
            df_feat['T02M_mean'].shift(lag)
        )
    df_feat['T02M_roll3_mean'] = (
        df_feat['T02M_mean']
        .shift(1)
        .rolling(window=3)
        .mean()
    )

    # Conserva el calendario completo al calcular los rezagos.
    df_feat = df_feat.reindex(daily_data.index)

    return df_feat.dropna().copy(), coef


COLUMNAS_EXCLUIDAS = [
    'SIN_max',
    'trend',
    'target_detrended',
    'time_idx',
    'ISHOLIDAY_max'
]

print("Constructor de features listo.")

Constructor de features listo.


## Bloque 4 — Optimizar hiperparámetros con Optuna

Optuna entrena hasta el 31/12/2017 y valida durante 2018–2019. Luego se congelan esos hiperparámetros para el walk-forward.

In [4]:
FIN_TRAIN = pd.Timestamp('2017-12-31')
FIN_VALIDACION = pd.Timestamp('2019-12-31')
INICIO_CALIBRACION = pd.Timestamp('2018-01-01')
INICIO_TEST = pd.Timestamp('2020-01-01')
FIN_TEST_EXCLUSIVO = min(
    df_daily.index.max().normalize() + pd.Timedelta(days=1),
    pd.Timestamp('2023-01-01')
)

ANIOS_VENTANA = 10
MESES_REENTRENAMIENTO = 6

df_tune, _ = build_features_final(df_daily, FIN_TRAIN)

features = [
    columna for columna in df_tune.columns
    if columna not in COLUMNAS_EXCLUIDAS
]

inicio_ventana_tune = (
    FIN_TRAIN -
    pd.DateOffset(years=ANIOS_VENTANA) +
    pd.Timedelta(days=1)
)

train_tune = df_tune.loc[
    (df_tune.index >= inicio_ventana_tune) &
    (df_tune.index <= FIN_TRAIN)
]
val_tune = df_tune.loc[
    (df_tune.index > FIN_TRAIN) &
    (df_tune.index <= FIN_VALIDACION)
]

if train_tune.empty or val_tune.empty:
    raise ValueError(
        "No se formó el entrenamiento hasta 2017 y la validación 2018–2019. "
        "Revisa las fechas disponibles."
    )

X_train = train_tune[features]
y_train = train_tune['target_detrended']
X_val = val_tune[features]

def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'n_estimators': trial.suggest_int('n_estimators', 150, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float(
            'colsample_bytree', 0.6, 1.0
        ),
        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-3, 10.0, log=True
        ),
        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-3, 10.0, log=True
        ),
        'random_state': 42,
        'n_jobs': -1
    }

    modelo = xgb.XGBRegressor(**params)
    modelo.fit(X_train, y_train, verbose=False)

    pred_val_detrended = modelo.predict(X_val)
    pred_val_mw = (
        pred_val_detrended *
        val_tune['trend'].to_numpy()
    )

    return mean_absolute_error(
        val_tune['SIN_max'].to_numpy(),
        pred_val_mw
    )

print("Optimizando hiperparámetros con train hasta 2017 y validación 2018–2019...")
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42)
)
study.optimize(objective, n_trials=50)

best_params = study.best_params.copy()
best_params.update({
    'objective': 'reg:absoluteerror',
    'random_state': 42,
    'n_jobs': -1
})

print("Mejores hiperparámetros:")
print(best_params)
print(f"Filas de entrenamiento: {len(train_tune)}")
print(f"Filas de validación: {len(val_tune)}")

Optimizando hiperparámetros con train hasta 2017 y validación 2018–2019...
Mejores hiperparámetros:
{'n_estimators': 494, 'max_depth': 4, 'learning_rate': 0.08942126767312061, 'subsample': 0.741959332532651, 'colsample_bytree': 0.9482440934161143, 'reg_alpha': 0.01882296124901949, 'reg_lambda': 4.439267989946528, 'objective': 'reg:absoluteerror', 'random_state': 42, 'n_jobs': -1}
Filas de entrenamiento: 3625
Filas de validación: 730


## Bloque 5 — Walk-forward semestral

Genera predicciones base para 2018–2019 (calibración de la corrección) y 2020–2022 (evaluación final). Cada semestre se entrena únicamente con datos disponibles hasta el día anterior, usando una ventana móvil de hasta 10 años.

In [5]:
def predecir_por_semestres(fecha_inicio, fecha_fin_exclusiva):
    resultados = []
    features_resultado = []
    reentrenamientos = 0
    inicio_periodo = fecha_inicio

    while inicio_periodo < fecha_fin_exclusiva:
        fin_periodo = min(
            inicio_periodo +
            pd.DateOffset(months=MESES_REENTRENAMIENTO),
            fecha_fin_exclusiva
        )
        fecha_corte = inicio_periodo - pd.Timedelta(days=1)

        df_fold, _ = build_features_final(df_daily, fecha_corte)

        inicio_ventana = (
            fecha_corte -
            pd.DateOffset(years=ANIOS_VENTANA) +
            pd.Timedelta(days=1)
        )

        train_fold = df_fold.loc[
            (df_fold.index >= inicio_ventana) &
            (df_fold.index <= fecha_corte)
        ]
        test_fold = df_fold.loc[
            (df_fold.index >= inicio_periodo) &
            (df_fold.index < fin_periodo)
        ]

        if train_fold.empty or test_fold.empty:
            raise ValueError(
                f"No se pudo generar el semestre desde {inicio_periodo.date()}."
            )

        modelo = xgb.XGBRegressor(**best_params)
        modelo.fit(
            train_fold[features],
            train_fold['target_detrended'],
            verbose=False
        )

        pred_detrended = modelo.predict(test_fold[features])
        pred_mw = pred_detrended * test_fold['trend'].to_numpy()
        fechas = test_fold.index.normalize()

        semestre = [
            f"{fecha.year}-S{1 if fecha.month <= 6 else 2}"
            for fecha in fechas
        ]

        df_pred = pd.DataFrame({
            'FECHA': fechas,
            'SIN_REAL': test_fold['SIN_max'].to_numpy(),
            'SIN_PREDICHO_BASE': pred_mw,
            'REENTRENADO_HASTA': fecha_corte,
            'SEMESTRE': semestre
        })
        resultados.append(df_pred)

        df_features = test_fold[features].copy()
        df_features.insert(0, 'FECHA', fechas)
        df_features['SIN_REAL'] = test_fold['SIN_max'].to_numpy()
        df_features['SIN_PREDICHO_BASE'] = pred_mw
        df_features['ERROR_BASE_MW'] = (
            df_features['SIN_REAL'] -
            df_features['SIN_PREDICHO_BASE']
        )
        df_features['REENTRENADO_HASTA'] = fecha_corte
        df_features['SEMESTRE'] = semestre
        features_resultado.append(df_features)

        reentrenamientos += 1
        print(
            f"Predicho {inicio_periodo:%Y-%m-%d} a "
            f"{(fin_periodo - pd.Timedelta(days=1)):%Y-%m-%d}"
        )

        inicio_periodo = fin_periodo

    return (
        pd.concat(resultados, ignore_index=True)
          .sort_values('FECHA')
          .reset_index(drop=True),
        pd.concat(features_resultado, ignore_index=True)
          .sort_values('FECHA')
          .reset_index(drop=True),
        reentrenamientos
    )


# Predicciones para calibrar la corrección: 2018–2019
df_calibracion_base, _, reentrenamientos_cal = predecir_por_semestres(
    INICIO_CALIBRACION,
    INICIO_TEST
)

# Predicciones que se evaluarán: 2020–2022
df_test_base, df_features_test, reentrenamientos_test = predecir_por_semestres(
    INICIO_TEST,
    FIN_TEST_EXCLUSIVO
)

print(f"Reentrenamientos 2018–2019: {reentrenamientos_cal}")
print(f"Reentrenamientos 2020–2022: {reentrenamientos_test}")
print(f"Días de evaluación 2020–2022: {len(df_test_base)}")

Predicho 2018-01-01 a 2018-06-30
Predicho 2018-07-01 a 2018-12-31
Predicho 2019-01-01 a 2019-06-30
Predicho 2019-07-01 a 2019-12-31
Predicho 2020-01-01 a 2020-06-30
Predicho 2020-07-01 a 2020-12-31
Predicho 2021-01-01 a 2021-06-30
Predicho 2021-07-01 a 2021-12-31
Predicho 2022-01-01 a 2022-06-30
Predicho 2022-07-01 a 2022-12-31
Reentrenamientos 2018–2019: 4
Reentrenamientos 2020–2022: 6
Días de evaluación 2020–2022: 1096


## Bloque 6 — Calibrar la corrección en 2018–2019 y evaluar en 2020–2022

La selección de ventana y alpha usa solamente los errores previos de 2018–2019. Durante la evaluación, cada día utiliza errores conocidos hasta el día anterior.

In [6]:
def calcular_metricas(real, predicho):
    real = np.asarray(real, dtype=float)
    predicho = np.asarray(predicho, dtype=float)
    error = real - predicho
    denominador = np.where(real == 0, np.nan, np.abs(real))

    return {
        'N': len(real),
        'MAE_MW': np.nanmean(np.abs(error)),
        'RMSE_MW': np.sqrt(np.nanmean(error ** 2)),
        'MAPE_pct': np.nanmean(
            np.abs(error) / denominador * 100
        )
    }


cal = df_calibracion_base.copy()
cal['ERROR_BASE'] = cal['SIN_REAL'] - cal['SIN_PREDICHO_BASE']

candidatos = []

for ventana in [3, 7, 14]:
    sesgo_previo = (
        cal['ERROR_BASE']
        .shift(1)
        .rolling(window=ventana, min_periods=1)
        .mean()
        .fillna(0)
    )

    for alpha in [0.25, 0.50, 0.75, 1.00]:
        pred_ajustada = (
            cal['SIN_PREDICHO_BASE'] +
            alpha * sesgo_previo
        )

        metricas = calcular_metricas(
            cal['SIN_REAL'],
            pred_ajustada
        )

        candidatos.append({
            'VENTANA_DIAS': ventana,
            'ALPHA': alpha,
            **metricas
        })

tabla_calibracion = (
    pd.DataFrame(candidatos)
    .sort_values(['MAE_MW', 'RMSE_MW'])
    .reset_index(drop=True)
)

mejor_ventana = int(tabla_calibracion.loc[0, 'VENTANA_DIAS'])
mejor_alpha = float(tabla_calibracion.loc[0, 'ALPHA'])

print(
    f"Corrección seleccionada en 2018–2019: "
    f"ventana={mejor_ventana} días, alpha={mejor_alpha:.2f}"
)
print("Mejores opciones en calibración:")
print(
    tabla_calibracion.head(5)
    .round(2)
    .to_string(index=False)
)

# Unir calibración y prueba en orden cronológico.
# El error del día actual se usa para corregir días siguientes.
df_todos = pd.concat(
    [
        cal.assign(PERIODO='CALIBRACION'),
        df_test_base.assign(PERIODO='EVALUACION')
    ],
    ignore_index=True
).sort_values('FECHA').reset_index(drop=True)

df_todos['ERROR_BASE'] = (
    df_todos['SIN_REAL'] -
    df_todos['SIN_PREDICHO_BASE']
)

df_todos['SESGO_PREVIO_MW'] = (
    df_todos['ERROR_BASE']
    .shift(1)
    .rolling(window=mejor_ventana, min_periods=1)
    .mean()
    .fillna(0)
)

df_todos['SIN_PREDICHO_AJUSTADO'] = (
    df_todos['SIN_PREDICHO_BASE'] +
    mejor_alpha * df_todos['SESGO_PREVIO_MW']
)

df_eval = df_todos.loc[
    df_todos['PERIODO'] == 'EVALUACION'
].copy()

df_eval['ERROR_AJUSTADO_MW'] = (
    df_eval['SIN_REAL'] -
    df_eval['SIN_PREDICHO_AJUSTADO']
)
df_eval['ERROR_ABS_BASE_MW'] = df_eval['ERROR_BASE'].abs()
df_eval['ERROR_ABS_AJUSTADO_MW'] = (
    df_eval['ERROR_AJUSTADO_MW'].abs()
)

metricas_base = calcular_metricas(
    df_eval['SIN_REAL'],
    df_eval['SIN_PREDICHO_BASE']
)
metricas_ajustado = calcular_metricas(
    df_eval['SIN_REAL'],
    df_eval['SIN_PREDICHO_AJUSTADO']
)

print("\n" + "=" * 62)
print("RESULTADOS FINALES WALK-FORWARD 2020–2022")
print(f"Reentrenamientos semestrales: {reentrenamientos_test}")
print(f"Días evaluados: {len(df_eval)}")
print("=" * 62)

resumen_global = pd.DataFrame([
    {'MODELO': 'Sin corrección', **metricas_base},
    {
        'MODELO': f'Corrección {mejor_ventana} días, alpha {mejor_alpha:.2f}',
        **metricas_ajustado
    }
])
print("\nMétricas globales:")
print(resumen_global.round(2).to_string(index=False))

resumen_semestres = []

for semestre, grupo in df_eval.groupby('SEMESTRE', sort=True):
    base = calcular_metricas(
        grupo['SIN_REAL'],
        grupo['SIN_PREDICHO_BASE']
    )
    ajustado = calcular_metricas(
        grupo['SIN_REAL'],
        grupo['SIN_PREDICHO_AJUSTADO']
    )

    resumen_semestres.append({
        'SEMESTRE': semestre,
        'N': len(grupo),
        'MAE_BASE': base['MAE_MW'],
        'MAE_AJUSTADO': ajustado['MAE_MW'],
        'RMSE_BASE': base['RMSE_MW'],
        'RMSE_AJUSTADO': ajustado['RMSE_MW'],
        'MAPE_BASE': base['MAPE_pct'],
        'MAPE_AJUSTADO': ajustado['MAPE_pct']
    })

df_metricas_semestres = pd.DataFrame(resumen_semestres)

print("\nMétricas por semestre:")
print(df_metricas_semestres.round(2).to_string(index=False))

Corrección seleccionada en 2018–2019: ventana=7 días, alpha=0.50
Mejores opciones en calibración:
 VENTANA_DIAS  ALPHA   N  MAE_MW  RMSE_MW  MAPE_pct
            7   0.50 730   91.06   125.54      3.74
            7   0.25 730   91.29   125.98      3.75
           14   0.50 730   91.36   125.91      3.75
           14   0.25 730   91.59   126.48      3.76
           14   0.75 730   91.87   126.73      3.77

RESULTADOS FINALES WALK-FORWARD 2020–2022
Reentrenamientos semestrales: 6
Días evaluados: 1096

Métricas globales:
                       MODELO    N  MAE_MW  RMSE_MW  MAPE_pct
               Sin corrección 1096   92.30   128.34      3.37
Corrección 7 días, alpha 0.50 1096   88.52   121.73      3.26

Métricas por semestre:
SEMESTRE   N  MAE_BASE  MAE_AJUSTADO  RMSE_BASE  RMSE_AJUSTADO  MAPE_BASE  MAPE_AJUSTADO
 2020-S1 182     90.00         86.17     122.61         119.31       3.49           3.33
 2020-S2 184     92.50         92.51     130.71         128.56       3.44           3.

## Bloque 7 — Exportar predicciones, features y peores fechas

El primer Excel tendrá exactamente tres columnas con la predicción corregida. El segundo conservará las features del modelo, ambas predicciones y sus errores; incluirá otra hoja con los 20 días de mayor error ajustado.

In [7]:
# Vincular predicciones ajustadas con las features del período de prueba.
columnas_resultado = [
    'FECHA',
    'SIN_PREDICHO_AJUSTADO',
    'SESGO_PREVIO_MW',
    'ERROR_BASE',
    'ERROR_AJUSTADO_MW',
    'ERROR_ABS_BASE_MW',
    'ERROR_ABS_AJUSTADO_MW'
]

df_features_final = df_features_test.merge(
    df_eval[columnas_resultado],
    on='FECHA',
    how='left'
)

# Excel de comparación con otros modelos: exactamente tres columnas.
df_excel_predicciones = df_eval[
    ['FECHA', 'SIN_REAL', 'SIN_PREDICHO_AJUSTADO']
].rename(columns={
    'SIN_PREDICHO_AJUSTADO': 'SIN_PREDICHO'
})

archivo_predicciones = 'predicciones_modelo_definitivo_2020_2022.xlsx'
df_excel_predicciones.to_excel(
    archivo_predicciones,
    index=False
)

# 20 días con mayor error absoluto después de la corrección.
columnas_diagnostico = [
    'FECHA',
    'SEMESTRE',
    'SIN_REAL',
    'SIN_PREDICHO_BASE',
    'SIN_PREDICHO_AJUSTADO',
    'ERROR_BASE_MW',
    'ERROR_AJUSTADO_MW',
    'T02M_mean',
    'T02M_max',
    'T02M_min',
    'T02M_range',
    'RH2M_mean',
    'RH2M_max',
    'WIND10M_mean',
    'WIND10M_max',
    'target_detrended_lag_21',
    'target_detrended_lag_28',
    'is_py_holiday',
    'is_holiday_eve',
    'receso_escolar'
]
columnas_diagnostico = [
    columna for columna in columnas_diagnostico
    if columna in df_features_final.columns
]

df_peores_20 = (
    df_features_final
    .nlargest(20, 'ERROR_ABS_AJUSTADO_MW')[columnas_diagnostico]
    .reset_index(drop=True)
)

archivo_features = 'features_y_errores_modelo_definitivo_2020_2022.xlsx'
with pd.ExcelWriter(archivo_features, engine='openpyxl') as writer:
    df_features_final.to_excel(
        writer,
        sheet_name='Features_y_errores',
        index=False
    )
    df_peores_20.to_excel(
        writer,
        sheet_name='20_peores_dias',
        index=False
    )

print("\nArchivos creados:")
print(archivo_predicciones)
print(archivo_features)

print("\n20 días con mayor error ajustado:")
print(df_peores_20.round(2).to_string(index=False))

files.download(archivo_predicciones)
files.download(archivo_features)


Archivos creados:
predicciones_modelo_definitivo_2020_2022.xlsx
features_y_errores_modelo_definitivo_2020_2022.xlsx

20 días con mayor error ajustado:
     FECHA SEMESTRE  SIN_REAL  SIN_PREDICHO_BASE  SIN_PREDICHO_AJUSTADO  ERROR_BASE_MW  ERROR_AJUSTADO_MW  T02M_mean  T02M_max  T02M_min  T02M_range  RH2M_mean  RH2M_max  WIND10M_mean  WIND10M_max  target_detrended_lag_21  target_detrended_lag_28  is_py_holiday  is_holiday_eve  receso_escolar
2022-12-11  2022-S2    3240.0            2642.34                2724.83         597.66             515.17      24.97     26.77     23.43        3.34      90.12     96.30          2.45         5.26                     1.09                     0.79              0               0               1
2020-10-26  2020-S2    3093.0            2573.27                2639.96         519.73             453.04      24.58     29.82     20.14        9.68      76.30     95.93          3.11         6.42                     0.91                     0.90              

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notas para interpretar los resultados

- Optuna usa 2018–2019 para seleccionar hiperparámetros; por eso las métricas finales se reportan en 2020–2022.
- La corrección se selecciona con residuos 2018–2019 y queda fija para 2020–2022.
- En cada día de prueba, la corrección móvil usa únicamente errores observados hasta el día anterior.
- El archivo histórico contiene meteorología observada. Si se quiere estimar el rendimiento operativo con pronósticos meteorológicos, hay que sustituir esas columnas por pronósticos archivados disponibles antes de cada fecha.